# Combined Transcriptomics Cleaning
Processes 2019_UGA, 2020_UGA, and SDY2867 with a shared pipeline, then concatenates into a single parquet.

In [1]:
import os
import pandas as pd

from data_cleaning.utils import log_standard_scale, peek

DATA_PATH = "../../data"
CLEAN_DATA_PATH = "../../cleaned_data"
MIN_PRESENT = 1  # drop any column with less than % non-NaN values

In [2]:
unique_genes_challenge = set(
    pd.read_csv(DATA_PATH + '/challenge_transcriptomics.tsv', sep='\t')['ensembl_gene_id'].unique()
)
print(f"Challenge genes: {len(unique_genes_challenge)}")

Challenge genes: 54902


In [3]:
filenames = [
    'train_transcriptomics_2019_UGA.tsv',
    'train_transcriptomics_2020_UGA.tsv',
    'train_transcriptomics_SDY2867.tsv',
]
df_raw = pd.concat(
    [pd.read_csv(DATA_PATH + '/' + f, sep='\t') for f in filenames],
    axis=0
).reset_index(drop=True)
print(f"Combined raw: {df_raw.shape}")
df_raw.head()

Combined raw: (56162050, 7)


,transcriptomics_id,participant_id,timepoint,ensembl_gene_id,raw_count,tpm_count,material
0,BS_RNA__2019_UGA.ID_208__2019_UGA_Standard_Flu...,2019_UGA.ID_208,0,ENSG00000000003,NaN,0.077926,Unknown
1,BS_RNA__2019_UGA.ID_208__2019_UGA_Standard_Flu...,2019_UGA.ID_208,0,ENSG00000000005,NaN,0.000000,Unknown
2,BS_RNA__2019_UGA.ID_208__2019_UGA_Standard_Flu...,2019_UGA.ID_208,0,ENSG00000000419,NaN,5.342548,Unknown
3,BS_RNA__2019_UGA.ID_208__2019_UGA_Standard_Flu...,2019_UGA.ID_208,0,ENSG00000000457,NaN,9.799336,Unknown
4,BS_RNA__2019_UGA.ID_208__2019_UGA_Standard_Flu...,2019_UGA.ID_208,0,ENSG00000000460,NaN,2.118530,Unknown


In [4]:
# Basic preprocessing, drop unused cols, and genes not in common with the challenge set
df = df_raw.drop(columns=['transcriptomics_id', 'raw_count', 'material'])
df = df[df['timepoint'].isin([
    0
    # 7  # too much missingness
])]
df = df[df['ensembl_gene_id'].isin(unique_genes_challenge)]
print(len(df))
df.head()

14244194


,participant_id,timepoint,ensembl_gene_id,tpm_count
0,2019_UGA.ID_208,0,ENSG00000000003,0.077926
1,2019_UGA.ID_208,0,ENSG00000000005,0.000000
2,2019_UGA.ID_208,0,ENSG00000000419,5.342548
3,2019_UGA.ID_208,0,ENSG00000000457,9.799336
4,2019_UGA.ID_208,0,ENSG00000000460,2.118530


In [5]:
df_pivot = df.pivot_table(
    index='participant_id',
    columns=['timepoint', 'ensembl_gene_id'],
    values='tpm_count'
)
df_pivot.columns = [f'TRAN_{gene}_d{int(tp)}' for tp, gene in df_pivot.columns]
df_pivot = df_pivot.reset_index()

In [6]:
peek(df_pivot)

,participant_id,TRAN_ENSG00000000003_d0,TRAN_ENSG00000000005_d0,TRAN_ENSG00000000419_d0,TRAN_ENSG00000000457_d0,TRAN_ENSG00000000460_d0,TRAN_ENSG00000000938_d0,TRAN_ENSG00000000971_d0,TRAN_ENSG00000001036_d0,TRAN_ENSG00000001084_d0,TRAN_ENSG00000001167_d0,TRAN_ENSG00000001460_d0,TRAN_ENSG00000001461_d0,TRAN_ENSG00000001497_d0,TRAN_ENSG00000001561_d0,TRAN_ENSG00000001617_d0,TRAN_ENSG00000001626_d0,TRAN_ENSG00000001629_d0,TRAN_ENSG00000001630_d0,TRAN_ENSG00000001631_d0
0,2019_UGA.ID_001,0.149866,0.0,8.066946,8.457258,3.973795,95.386306,0.034801,13.583601,7.965179,7.187047,1.590390,13.973659,9.685137,3.203689,0.000000,0.0,10.685926,3.141505,8.707056
1,2019_UGA.ID_005,0.119232,0.0,3.209018,7.973900,2.521538,83.430649,0.110751,10.308613,12.190398,9.965058,1.533703,16.946756,10.774215,6.274083,0.029305,0.0,16.372074,2.892793,11.885934
2,2019_UGA.ID_008,0.096349,0.0,5.277265,7.289030,1.913696,138.440845,0.143194,10.557651,14.385306,5.401848,1.982958,9.914634,6.998915,2.006857,0.000000,0.0,4.988143,2.244099,6.075266
3,2019_UGA.ID_011,0.044666,0.0,3.733021,8.731205,3.491700,150.812198,0.074682,11.532723,14.860079,11.703771,1.149111,10.513365,6.511664,10.136052,0.000000,0.0,15.577613,2.566189,12.220556
4,2019_UGA.ID_014,0.000000,0.0,5.207440,9.466478,2.931771,129.608966,0.128051,15.970298,13.050625,10.539603,1.994948,15.887030,8.589723,10.012239,0.112938,0.0,14.760842,3.121698,10.802686


In [7]:
gene_cols = [c for c in df_pivot.columns if c != 'participant_id']
present_frac = df_pivot[gene_cols].notna().mean()
sparse_cols = present_frac[present_frac < MIN_PRESENT].index.tolist()
df_pivot = df_pivot.drop(columns=sparse_cols)
print(f'Dropped {len(sparse_cols)} columns with <{MIN_PRESENT:.0%} present values.')
print(f'Shape after dropping: {df_pivot.shape}')

Dropped 24080 columns with <100% present values.
Shape after dropping: (395, 30823)


In [8]:
d0_genes = {c.replace('_d0', '') for c in df_pivot.columns if c.endswith('_d0')}
d7_genes = {c.replace('_d7', '') for c in df_pivot.columns if c.endswith('_d7')}
missing_d7 = d0_genes - d7_genes
print(f"d0 cols: {len(d0_genes)}, d7 cols: {len(d7_genes)}")
print(f"Genes with d0 but no d7: {len(missing_d7)}")

d0 cols: 30822, d7 cols: 0
Genes with d0 but no d7: 30822


In [9]:
# df_pivot = log_standard_scale(df_pivot)
# print(df_pivot.shape)
# peek(df_pivot)

In [10]:
os.makedirs(CLEAN_DATA_PATH, exist_ok=True)
df_pivot.to_parquet(CLEAN_DATA_PATH + '/transcriptomics_combined_cleaned.parquet', index=False)